[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/tools/notebooks/07_replay_and_verify.ipynb)

<div style="font-family:Arial,sans-serif">

# 07. Replay and verify

Upload the step 06 ZIP and reproduce the accepted extraction and adjustments.

Files move between your computer and Colab through the upload and download
buttons. Each step tells you which file to choose and what to save.

Created by **Mohsen Tahmasebi Nasab, PhD**<br>
[hydromohsen.com](https://hydromohsen.com)

Copyright and license holder: Mohsen Tahmasebi Nasab. Notebook code is
licensed under the repository's MIT License. Rang palette data follows the
CC0 dedication described in the licensing guide. Source images keep their own
rights and reuse terms.

</div>

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Give the palette a short filename. Use the same name in all seven notebooks.</div>

In [ ]:
PALETTE_SLUG = "your-palette" #@param {type:"string"}
DOWNLOAD_UPDATED_ZIP = True #@param {type:"boolean"}

In [ ]:
import json
import shutil
import subprocess
import sys
import types
import zipfile
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    pass

try:
    import matplotlib
    import numpy
    import PIL
    import sklearn
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pillow", "scikit-learn", "matplotlib"
    ], check=True)

MODULE_SOURCES = {"colorlib": "\"\"\"Shared color math for the Rang tools.\n\nCovers sRGB to CIELAB and LCh conversions, color vision deficiency simulation\nafter Machado et al. (2009), the CIEDE2000 color difference, and the palette\nlevel helpers built on top of them. Pure numpy, so the whole toolchain runs\nwithout a color science dependency.\n\"\"\"\nimport itertools\nimport json\nimport pathlib\nimport re\nimport urllib.request\n\nimport numpy as np\n\nREPO_ROOT = pathlib.Path(__file__).resolve().parent.parent\nPALETTE_DIR = REPO_ROOT / \"palettes\"\nCACHE_DIR = REPO_ROOT / \"cache\"\n\n# Project screening threshold for minimum pairwise CIEDE2000 under all four\n# viewing simulations. Use it as a comparison, then check the finished figure.\nCOLORBLIND_THRESHOLD = 8.0\n\n# ---------------------------------------------------------------- sRGB and Lab\n\nM_XYZ = np.array([[0.4124564, 0.3575761, 0.1804375],\n                  [0.2126729, 0.7151522, 0.0721750],\n                  [0.0193339, 0.1191920, 0.9503041]])\nWHITE = np.array([0.95047, 1.0, 1.08883])\n\n\ndef hex_to_rgb(h):\n    h = h.lstrip(\"#\")\n    return np.array([int(h[i:i + 2], 16) for i in (0, 2, 4)], float)\n\n\ndef rgb_to_hex(rgb):\n    r, g, b = np.round(np.clip(rgb, 0, 255)).astype(int)\n    return f\"#{r:02x}{g:02x}{b:02x}\"\n\n\ndef hex_to_argb_int(hexcode, alpha=255):\n    \"\"\"The signed 32 bit ARGB integer .NET uses, as found in .rasmap files.\n\n    White with full alpha is -1, pure yellow is -256, which matches the\n    Colors attribute RAS Mapper writes in SurfaceFill elements.\n    \"\"\"\n    if isinstance(alpha, bool) or not isinstance(alpha, int) or not 0 <= alpha <= 255:\n        raise ValueError(\"alpha must be an integer from 0 to 255\")\n    r, g, b = (int(v) for v in hex_to_rgb(hexcode))\n    value = (alpha << 24) | (r << 16) | (g << 8) | b\n    return value - 2 ** 32 if value >= 2 ** 31 else value\n\n\ndef srgb_to_linear(c):\n    c = np.asarray(c, float) / 255.0\n    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)\n\n\ndef linear_to_srgb(c):\n    c = np.clip(np.asarray(c, float), 0, 1)\n    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055) * 255\n\n\ndef rgb_to_lab(rgb):\n    xyz = srgb_to_linear(rgb) @ M_XYZ.T / WHITE\n    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)\n    return np.stack([116 * f[..., 1] - 16,\n                     500 * (f[..., 0] - f[..., 1]),\n                     200 * (f[..., 1] - f[..., 2])], -1)\n\n\ndef lab_to_rgb(lab):\n    lab = np.asarray(lab, float)\n    fy = (lab[..., 0] + 16) / 116\n    fx, fz = fy + lab[..., 1] / 500, fy - lab[..., 2] / 200\n    f = np.stack([fx, fy, fz], -1)\n    xyz = np.where(f ** 3 > 0.008856, f ** 3, (f - 16 / 116) / 7.787) * WHITE\n    return linear_to_srgb(xyz @ np.linalg.inv(M_XYZ).T)\n\n\ndef lab_to_lch(lab):\n    L, a, b = np.asarray(lab, float)\n    return np.array([L, np.hypot(a, b), np.degrees(np.arctan2(b, a)) % 360])\n\n\ndef lch_to_lab(lch):\n    L, C, h = np.asarray(lch, float)\n    return np.array([L, C * np.cos(np.radians(h)), C * np.sin(np.radians(h))])\n\n\n# ------------------------------------------- vision simulation (Machado 2009)\n# Severity 1.0 matrices applied in linear light RGB.\n\nCVD_MATRIX = {\n    \"protan\": np.array([[0.152286, 1.052583, -0.204868],\n                        [0.114503, 0.786281, 0.099216],\n                        [-0.003882, -0.048116, 1.051998]]),\n    \"deutan\": np.array([[0.367322, 0.860646, -0.227968],\n                        [0.280085, 0.672501, 0.047413],\n                        [-0.011820, 0.042940, 0.968881]]),\n    \"tritan\": np.array([[1.255528, -0.076749, -0.178779],\n                        [-0.078411, 0.930809, 0.147602],\n                        [0.004733, 0.691367, 0.303900]]),\n}\n\nVISION_TYPES = (None, \"protan\", \"deutan\", \"tritan\")\nVISION_LABELS = {None: \"normal vision\", \"protan\": \"protanopia\",\n                 \"deutan\": \"deuteranopia\", \"tritan\": \"tritanopia\"}\n\n\ndef simulate(rgb, kind):\n    \"\"\"Simulate dichromatic vision. kind=None returns rgb unchanged.\"\"\"\n    if kind is None:\n        return np.asarray(rgb, float)\n    return linear_to_srgb(srgb_to_linear(rgb) @ CVD_MATRIX[kind].T)\n\n\n# ------------------------------------------------------------------- CIEDE2000\n\ndef ciede2000(lab1, lab2):\n    L1, a1, b1 = lab1\n    L2, a2, b2 = lab2\n    C1, C2 = np.hypot(a1, b1), np.hypot(a2, b2)\n    Cb = (C1 + C2) / 2\n    G = 0.5 * (1 - np.sqrt(Cb ** 7 / (Cb ** 7 + 25 ** 7)))\n    a1p, a2p = (1 + G) * a1, (1 + G) * a2\n    C1p, C2p = np.hypot(a1p, b1), np.hypot(a2p, b2)\n    h1p = np.degrees(np.arctan2(b1, a1p)) % 360\n    h2p = np.degrees(np.arctan2(b2, a2p)) % 360\n    dLp, dCp = L2 - L1, C2p - C1p\n    if C1p * C2p == 0:\n        dhp = 0.0\n    elif abs(h2p - h1p) <= 180:\n        dhp = h2p - h1p\n    elif h2p - h1p > 180:\n        dhp = h2p - h1p - 360\n    else:\n        dhp = h2p - h1p + 360\n    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dhp) / 2)\n    Lbp, Cbp = (L1 + L2) / 2, (C1p + C2p) / 2\n    if C1p * C2p == 0:\n        hbp = h1p + h2p\n    elif abs(h1p - h2p) <= 180:\n        hbp = (h1p + h2p) / 2\n    elif h1p + h2p < 360:\n        hbp = (h1p + h2p + 360) / 2\n    else:\n        hbp = (h1p + h2p - 360) / 2\n    T = (1 - 0.17 * np.cos(np.radians(hbp - 30)) + 0.24 * np.cos(np.radians(2 * hbp))\n         + 0.32 * np.cos(np.radians(3 * hbp + 6)) - 0.20 * np.cos(np.radians(4 * hbp - 63)))\n    dth = 30 * np.exp(-(((hbp - 275) / 25) ** 2))\n    Rc = 2 * np.sqrt(Cbp ** 7 / (Cbp ** 7 + 25 ** 7))\n    Sl = 1 + (0.015 * (Lbp - 50) ** 2) / np.sqrt(20 + (Lbp - 50) ** 2)\n    Sc = 1 + 0.045 * Cbp\n    Sh = 1 + 0.015 * Cbp * T\n    Rt = -np.sin(np.radians(2 * dth)) * Rc\n    return float(np.sqrt((dLp / Sl) ** 2 + (dCp / Sc) ** 2 + (dHp / Sh) ** 2\n                         + Rt * (dCp / Sc) * (dHp / Sh)))\n\n\n# ---------------------------------------------------------- palette level math\n\ndef pairwise_min_mean(hexes, kind=None):\n    \"\"\"Minimum and mean CIEDE2000 between all color pairs under one vision type.\"\"\"\n    if len(hexes) < 2:\n        raise ValueError(\"at least two colors are required\")\n    lab = rgb_to_lab(simulate(np.array([hex_to_rgb(h) for h in hexes]), kind))\n    ds = [ciede2000(lab[i], lab[j]) for i, j in itertools.combinations(range(len(lab)), 2)]\n    return min(ds), float(np.mean(ds))\n\n\ndef worst_case(hexes):\n    \"\"\"Smallest pairwise distance across all four vision types.\"\"\"\n    return min(pairwise_min_mean(hexes, k)[0] for k in VISION_TYPES)\n\n\ndef worst_pair(hexes):\n    \"\"\"The single closest pair of colors and the vision type where it happens.\"\"\"\n    best = None\n    for kind in VISION_TYPES:\n        lab = rgb_to_lab(simulate(np.array([hex_to_rgb(h) for h in hexes]), kind))\n        for i, j in itertools.combinations(range(len(hexes)), 2):\n            d = ciede2000(lab[i], lab[j])\n            if best is None or d < best[0]:\n                best = (d, hexes[i], hexes[j], VISION_LABELS[kind])\n    return best\n\n\ndef greedy_order(hexes):\n    \"\"\"Build the discrete pick order stored with each palette.\n\n    order[i] is the n at which color i first enters the discrete palette, so\n    requesting n colors returns those with order <= n, kept in ramp sequence.\n    Each step adds the color that maximizes the minimum CIEDE2000 distance to\n    the colors already picked, judged across all four vision types at once.\n    \"\"\"\n    n = len(hexes)\n    rgb = np.array([hex_to_rgb(h) for h in hexes])\n    labs = {k: rgb_to_lab(simulate(rgb, k)) for k in VISION_TYPES}\n\n    def dist(i, j):\n        return min(ciede2000(labs[k][i], labs[k][j]) for k in VISION_TYPES)\n\n    first = max(itertools.combinations(range(n), 2), key=lambda p: dist(*p))\n    chosen = sorted(first)\n    while len(chosen) < n:\n        rest = [i for i in range(n) if i not in chosen]\n        chosen.append(max(rest, key=lambda i: min(dist(i, j) for j in chosen)))\n    order = [0] * n\n    for rank, i in enumerate(chosen, start=1):\n        order[i] = rank\n    return order\n\n\ndef discrete_subset(colors, order, n):\n    \"\"\"The n colors met by rank, kept in ramp sequence.\"\"\"\n    return [c for c, r in zip(colors, order) if r <= n]\n\n\ndef interpolate(colors, n):\n    \"\"\"n colors linearly interpolated along the ramp in sRGB.\"\"\"\n    if isinstance(n, bool) or not isinstance(n, int) or n < 1:\n        raise ValueError(\"n must be a positive integer\")\n    if not colors:\n        raise ValueError(\"at least one source color is required\")\n    if n == 1:\n        return [colors[0]]\n    rgbs = [hex_to_rgb(c) for c in colors]\n    out = []\n    for i in range(n):\n        t = i * (len(colors) - 1) / (n - 1)\n        j = min(int(t), len(colors) - 2)\n        f = t - j\n        out.append(rgb_to_hex(rgbs[j] + (rgbs[j + 1] - rgbs[j]) * f))\n    return out\n\n\ndef presence_in_image(hexes, image_path, step=17):\n    \"\"\"How close each color sits to a sampled point in the source photo.\n\n    The image is reduced to fit within 800 by 800 pixels, then every ``step``\n    point is evaluated. Returns each hex color's nearest CIEDE2000 distance\n    and the percentage of evaluated points within 8.0.\n    \"\"\"\n    from PIL import Image, ImageOps\n    if step < 1:\n        raise ValueError(\"step must be at least 1\")\n    im = ImageOps.exif_transpose(Image.open(image_path)).convert(\"RGB\")\n    im.thumbnail((800, 800), Image.Resampling.LANCZOS)\n    lab = rgb_to_lab(np.asarray(im).reshape(-1, 3).astype(float))[::step]\n    out = {}\n    for h in hexes:\n        target = rgb_to_lab(hex_to_rgb(h))\n        d = np.array([ciede2000(target, px) for px in lab])\n        out[h] = (float(d.min()), float((d < 8).mean() * 100))\n    return out\n\n\n# ------------------------------------------------------------ palette loading\n\ndef load_palette(name):\n    \"\"\"Read palettes/<name>.json. Accepts 'Kashan', 'kashan' or a file path.\"\"\"\n    p = pathlib.Path(name)\n    if not p.suffix == \".json\":\n        p = PALETTE_DIR / f\"{str(name).lower()}.json\"\n    if not p.exists():\n        options = \", \".join(sorted(f.stem for f in PALETTE_DIR.glob(\"*.json\")))\n        raise FileNotFoundError(f\"No palette file {p}. Available: {options}\")\n    pal = json.loads(p.read_text(encoding=\"utf-8\"))\n    for key in (\"name\", \"persian\", \"pronunciation\", \"colors\", \"notes\", \"source\"):\n        if key not in pal:\n            raise KeyError(f\"{p} is missing the required key '{key}'\")\n    if not isinstance(pal[\"name\"], str) or not re.fullmatch(r\"[A-Z][A-Za-z]*\", pal[\"name\"]):\n        raise ValueError(f\"{p}: name must be one capitalized ASCII word\")\n    if pal[\"name\"].lower() != p.stem.lower():\n        raise ValueError(f\"{p}: name must match the filename\")\n    colors = pal[\"colors\"]\n    if not isinstance(colors, list) or not 5 <= len(colors) <= 12:\n        raise ValueError(f\"{p}: colors must contain 5 to 12 values\")\n    if any(not isinstance(c, str) or not re.fullmatch(r\"#[0-9a-fA-F]{6}\", c)\n           for c in colors):\n        raise ValueError(f\"{p}: every color must be a six digit hex value\")\n    if len({c.lower() for c in colors}) != len(colors):\n        raise ValueError(f\"{p}: colors must be unique\")\n    notes = pal[\"notes\"]\n    if (not isinstance(notes, list) or len(notes) != len(colors)\n            or any(not isinstance(note, str) or not note.strip() for note in notes)):\n        raise ValueError(f\"{p}: notes must contain one nonempty entry per color\")\n    if \"order\" in pal and sorted(pal[\"order\"]) != list(range(1, len(colors) + 1)):\n        raise ValueError(f\"{p}: order must be a permutation from 1 to the color count\")\n    if \"colorblind\" in pal and not isinstance(pal[\"colorblind\"], bool):\n        raise ValueError(f\"{p}: colorblind must be true or false\")\n    if \"position\" in pal and (isinstance(pal[\"position\"], bool)\n                              or not isinstance(pal[\"position\"], int)\n                              or pal[\"position\"] < 1):\n        raise ValueError(f\"{p}: position must be a positive integer\")\n    src = pal[\"source\"]\n    source_keys = (\"title\", \"date\", \"geography\", \"medium\", \"url\", \"image\",\n                   \"public_domain\")\n    missing = [key for key in source_keys if key not in src]\n    if missing:\n        raise KeyError(f\"{p}: source is missing {', '.join(missing)}\")\n    if any(not isinstance(src[key], str) or not src[key].strip()\n           for key in source_keys if key != \"public_domain\"):\n        raise ValueError(f\"{p}: required source text fields must be nonempty strings\")\n    if not isinstance(src[\"public_domain\"], bool):\n        raise ValueError(f\"{p}: source.public_domain must be true or false\")\n    if src[\"public_domain\"]:\n        missing = [key for key in (\"museum\", \"accession\") if not src.get(key)]\n    else:\n        missing = [key for key in (\"credit\", \"rights\") if not src.get(key)]\n    if missing:\n        raise KeyError(f\"{p}: source is missing {', '.join(missing)}\")\n    return pal\n\n\ndef all_palettes():\n    \"\"\"Every palette, in gallery order.\n\n    Palettes carry a position number assigned in order of addition, so the\n    gallery keeps the collection's history instead of shuffling alphabetically\n    whenever a new name lands.\n    \"\"\"\n    pals = [load_palette(f.stem) for f in sorted(PALETTE_DIR.glob(\"*.json\"))]\n    return sorted(pals, key=lambda p: (p.get(\"position\", 10 ** 9), p[\"name\"]))\n\n\ndef fetch_image(url_or_path):\n    \"\"\"Return a local path for an image, downloading into cache/ when needed.\n\n    Accepts a URL, an absolute path, or a path relative to the repo root, which\n    is how palette files reference photos committed under sources/.\n    \"\"\"\n    p = pathlib.Path(url_or_path)\n    if p.exists():\n        return p\n    rel = REPO_ROOT / str(url_or_path)\n    if rel.exists():\n        return rel\n    CACHE_DIR.mkdir(exist_ok=True)\n    local = CACHE_DIR / pathlib.Path(str(url_or_path).split(\"?\")[0]).name\n    if not local.exists():\n        print(f\"downloading {url_or_path}\")\n        req = urllib.request.Request(str(url_or_path),\n                                     headers={\"User-Agent\": \"Mozilla/5.0\"})\n        with urllib.request.urlopen(req) as r, open(local, \"wb\") as f:\n            f.write(r.read())\n    return local\n", "adjust_colors": "\"\"\"Nudge palette colors in LCh when a sampled color is not quite right.\n\nEdits happen in lightness (L), chroma (C) and hue angle (H), which track how\npeople actually describe a color being off. Brighter or darker, more or less\nsaturated, leaning too orange. Position numbers are 1 based.\n\nExamples\n\n  python tools/adjust_colors.py --colors \"#8a9463,#345f72\" --edit \"1:L+4\"\n  python tools/adjust_colors.py --colors \"#7f3020,#ab4a47,#c07049\" ^\n      --edit \"2:L-3,C+5\" --edit \"3:H+8\" --png cache/before_after.png\n\nPrints the edited hex codes and, with --png, writes a before and after strip\nso you can judge the change against the original.\n\"\"\"\nimport argparse\nimport re\n\nfrom PIL import Image\n\nfrom colorlib import hex_to_rgb, lab_to_lch, lab_to_rgb, lch_to_lab, rgb_to_hex, rgb_to_lab\n\n\ndef parse_edit(text):\n    m = re.fullmatch(r\"(\\d+):(.+)\", text.strip())\n    if not m:\n        raise argparse.ArgumentTypeError(\"expected position:changes, like 2:L+5,C-3\")\n    idx = int(m.group(1)) - 1\n    changes = {}\n    for part in m.group(2).split(\",\"):\n        pm = re.fullmatch(r\"\\s*([LCHlch])\\s*([+-]\\d+(?:\\.\\d+)?)\\s*\", part)\n        if not pm:\n            raise argparse.ArgumentTypeError(f\"bad change '{part}', use L+5 C-3 H+10\")\n        changes[pm.group(1).upper()] = float(pm.group(2))\n    return idx, changes\n\n\ndef apply_edit(hexcode, changes):\n    lch = lab_to_lch(rgb_to_lab(hex_to_rgb(hexcode)))\n    lch[0] = max(0, min(100, lch[0] + changes.get(\"L\", 0)))\n    lch[1] = max(0, lch[1] + changes.get(\"C\", 0))\n    lch[2] = (lch[2] + changes.get(\"H\", 0)) % 360\n    return rgb_to_hex(lab_to_rgb(lch_to_lab(lch)))\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument(\"--colors\", required=True, help=\"comma separated hex codes\")\n    ap.add_argument(\"--edit\", action=\"append\", type=parse_edit, default=[],\n                    metavar=\"pos:changes\", help=\"edit one color, repeatable\")\n    ap.add_argument(\"--png\", help=\"optional before and after strip\")\n    args = ap.parse_args()\n\n    before = [c.strip() for c in args.colors.split(\",\") if c.strip()]\n    if not before:\n        ap.error(\"provide at least one color\")\n    after = list(before)\n    for idx, changes in args.edit:\n        if not 0 <= idx < len(before):\n            raise SystemExit(f\"position {idx + 1} is outside the palette\")\n        after[idx] = apply_edit(after[idx], changes)\n\n    print(\"before:\", \" \".join(before))\n    print(\"after: \", \" \".join(after))\n\n    if args.png:\n        cw, ch = 120, 90\n        img = Image.new(\"RGB\", (cw * len(before), ch * 2 + 8), \"white\")\n        for i, (b, a) in enumerate(zip(before, after)):\n            img.paste(Image.new(\"RGB\", (cw, ch), b), (i * cw, 0))\n            img.paste(Image.new(\"RGB\", (cw, ch), a), (i * cw, ch + 8))\n        img.save(args.png)\n        print(f\"wrote {args.png} (top row before, bottom row after)\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "notebook_workflow": "\"\"\"Shared functions for the numbered Rang contribution notebooks.\n\nCreated by Mohsen Tahmasebi Nasab, PhD\nhttps://hydromohsen.com\n\nCopyright (c) 2026 Mohsen Tahmasebi Nasab\nLicensed under the MIT License in the repository root.\n\"\"\"\nimport hashlib\nimport json\nimport pathlib\nimport platform\nimport re\nimport urllib.parse\nimport urllib.request\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport PIL\nimport sklearn\nfrom matplotlib.patches import Rectangle\nfrom PIL import Image, ImageOps\nfrom sklearn.cluster import KMeans\n\nfrom adjust_colors import apply_edit\nfrom colorlib import (COLORBLIND_THRESHOLD, VISION_LABELS, VISION_TYPES,\n                      greedy_order, hex_to_rgb, lab_to_rgb, pairwise_min_mean,\n                      presence_in_image, rgb_to_hex, rgb_to_lab, worst_case)\n\nCREATOR = \"Mohsen Tahmasebi Nasab, PhD\"\nWEBSITE = \"https://hydromohsen.com\"\nLICENSE_HOLDER = \"Mohsen Tahmasebi Nasab\"\nRECIPE_VERSION = 1\n\n\ndef use_arial():\n    \"\"\"Use Arial when installed and a close open fallback in Colab.\"\"\"\n    plt.rcParams.update({\n        \"font.family\": \"sans-serif\",\n        \"font.sans-serif\": [\"Arial\", \"Liberation Sans\", \"DejaVu Sans\"],\n        \"figure.dpi\": 120,\n        \"axes.spines.top\": False,\n        \"axes.spines.right\": False,\n    })\n\n\ndef palette_slug(name):\n    slug = re.sub(r\"[^a-z0-9]+\", \"-\", name.lower()).strip(\"-\")\n    if not slug:\n        raise ValueError(\"palette name must contain a letter or number\")\n    return slug\n\n\ndef read_json(path):\n    return json.loads(pathlib.Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef write_json(path, value):\n    path = pathlib.Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + \"\\n\",\n                    encoding=\"utf-8\")\n    return path\n\n\ndef _source_suffix(source):\n    suffix = pathlib.Path(urllib.parse.urlparse(str(source)).path).suffix.lower()\n    return suffix if suffix in {\".jpg\", \".jpeg\", \".png\", \".tif\", \".tiff\"} else \".img\"\n\n\ndef obtain_source(source, work_dir):\n    \"\"\"Return a local source path, downloading an HTTPS image when needed.\"\"\"\n    work_dir = pathlib.Path(work_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    source = str(source)\n    if source.startswith(\"http://\"):\n        raise ValueError(\"remote source images must use HTTPS\")\n    if source.startswith(\"https://\"):\n        path = work_dir / f\"source{_source_suffix(source)}\"\n        if not path.exists():\n            request = urllib.request.Request(source, headers={\"User-Agent\": \"Rang/1.0\"})\n            with urllib.request.urlopen(request, timeout=60) as response:\n                data = response.read(100 * 1024 * 1024 + 1)\n            if len(data) > 100 * 1024 * 1024:\n                raise ValueError(\"source image is larger than 100 MB\")\n            path.write_bytes(data)\n        return path\n    path = pathlib.Path(source).expanduser()\n    if not path.is_absolute():\n        path = pathlib.Path(work_dir) / path\n    if not path.exists():\n        raise FileNotFoundError(f\"source image was not found: {source}\")\n    return path.resolve()\n\n\ndef image_sha256(path):\n    digest = hashlib.sha256()\n    with pathlib.Path(path).open(\"rb\") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef open_rgb(path):\n    return ImageOps.exif_transpose(Image.open(path)).convert(\"RGB\")\n\n\ndef show_source(path, title=\"Source image\"):\n    use_arial()\n    image = open_rgb(path)\n    fig, ax = plt.subplots(figsize=(10, 12))\n    ax.imshow(image)\n    ax.set_title(f\"{title}, {image.width} x {image.height} pixels\")\n    ax.set_xlabel(\"x coordinate in pixels\")\n    ax.set_ylabel(\"y coordinate in pixels\")\n    ax.grid(color=\"white\", linewidth=0.5, alpha=0.35)\n    fig.tight_layout()\n    return fig\n\n\ndef _validate_regions(regions, width, height):\n    if not regions:\n        raise ValueError(\"define at least one region\")\n    seen = set()\n    cleaned = []\n    for region in regions:\n        region_id = str(region.get(\"id\", \"\")).strip()\n        if not re.fullmatch(r\"[a-z][a-z0-9-]*\", region_id):\n            raise ValueError(f\"bad region id: {region_id!r}\")\n        if region_id in seen:\n            raise ValueError(f\"duplicate region id: {region_id}\")\n        seen.add(region_id)\n        box = region.get(\"box\")\n        if (not isinstance(box, (list, tuple)) or len(box) != 4\n                or any(isinstance(value, bool) or not isinstance(value, int)\n                       for value in box)):\n            raise ValueError(f\"{region_id}: box must contain four pixel integers\")\n        x0, y0, x1, y1 = box\n        if x0 < 0 or y0 < 0 or x1 > width or y1 > height:\n            raise ValueError(f\"{region_id}: box lies outside the source image\")\n        if x1 <= x0 or y1 <= y0:\n            raise ValueError(f\"{region_id}: box is empty or reversed\")\n        k = int(region.get(\"k\", 8))\n        if not 2 <= k <= 20:\n            raise ValueError(f\"{region_id}: k must be from 2 to 20\")\n        cleaned.append({\n            \"id\": region_id,\n            \"label\": str(region.get(\"label\", region_id)).strip(),\n            \"box\": [x0, y0, x1, y1],\n            \"normalized\": [round(x0 / width, 6), round(y0 / height, 6),\n                           round(x1 / width, 6), round(y1 / height, 6)],\n            \"k\": k,\n            \"note\": str(region.get(\"note\", \"\")).strip(),\n        })\n    return cleaned\n\n\ndef create_recipe(palette_name, source, source_path, regions, recipe_path):\n    image = open_rgb(source_path)\n    recipe = {\n        \"schema_version\": RECIPE_VERSION,\n        \"palette\": palette_name,\n        \"creator\": CREATOR,\n        \"website\": WEBSITE,\n        \"license_holder\": LICENSE_HOLDER,\n        \"source\": {\n            \"image\": str(source),\n            \"sha256\": image_sha256(source_path),\n            \"oriented_width\": image.width,\n            \"oriented_height\": image.height,\n            \"color_handling\": \"EXIF orientation followed by RGB conversion\",\n        },\n        \"regions\": _validate_regions(regions, image.width, image.height),\n        \"extraction\": {\n            \"color_space\": \"CIELAB\",\n            \"white_point\": \"D65\",\n            \"algorithm\": \"sklearn.cluster.KMeans\",\n            \"random_state\": 0,\n            \"n_init\": 20,\n            \"algorithm_mode\": \"lloyd\",\n            \"maximum_region_size\": 400,\n        },\n        \"environment\": {\n            \"python\": platform.python_version(),\n            \"numpy\": np.__version__,\n            \"pillow\": PIL.__version__,\n            \"scikit_learn\": sklearn.__version__,\n        },\n    }\n    write_json(recipe_path, recipe)\n    return recipe\n\n\ndef region_overlay(recipe_path, work_dir, output=None):\n    use_arial()\n    recipe = read_json(recipe_path)\n    source_path = obtain_source(recipe[\"source\"][\"image\"], work_dir)\n    _verify_source(recipe, source_path)\n    image = open_rgb(source_path)\n    fig, ax = plt.subplots(figsize=(10, 12))\n    ax.imshow(image)\n    colors = plt.cm.tab10(np.linspace(0, 1, len(recipe[\"regions\"])))\n    for number, (region, color) in enumerate(zip(recipe[\"regions\"], colors), start=1):\n        x0, y0, x1, y1 = region[\"box\"]\n        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0,\n                               fill=False, edgecolor=color, linewidth=3))\n        ax.text(x0 + 12, y0 + 28, f'{number}. {region[\"label\"]}, k={region[\"k\"]}',\n                color=\"black\", fontsize=10,\n                bbox={\"facecolor\": \"white\", \"edgecolor\": color, \"alpha\": 0.9})\n    ax.set_title(f'{recipe[\"palette\"]} extraction regions')\n    ax.set_xlabel(\"x coordinate in pixels\")\n    ax.set_ylabel(\"y coordinate in pixels\")\n    fig.tight_layout()\n    if output:\n        output = pathlib.Path(output)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(output, facecolor=\"white\", bbox_inches=\"tight\")\n    return fig\n\n\ndef _verify_source(recipe, source_path):\n    actual = image_sha256(source_path)\n    expected = recipe[\"source\"][\"sha256\"]\n    if actual != expected:\n        raise ValueError(\"source checksum does not match the saved recipe\")\n    image = open_rgb(source_path)\n    size = (recipe[\"source\"][\"oriented_width\"],\n            recipe[\"source\"][\"oriented_height\"])\n    if image.size != size:\n        raise ValueError(\"source dimensions do not match the saved recipe\")\n\n\ndef _cluster_region(image, box, k, settings):\n    crop = image.crop(tuple(box))\n    crop.thumbnail((settings[\"maximum_region_size\"],\n                    settings[\"maximum_region_size\"]), Image.Resampling.LANCZOS)\n    pixels = np.asarray(crop).reshape(-1, 3).astype(float)\n    model = KMeans(\n        n_clusters=k,\n        n_init=settings[\"n_init\"],\n        random_state=settings[\"random_state\"],\n        algorithm=settings[\"algorithm_mode\"],\n    ).fit(rgb_to_lab(pixels))\n    counts = np.bincount(model.labels_, minlength=k)\n    rows = []\n    for index in range(k):\n        lab = model.cluster_centers_[index]\n        rows.append({\n            \"lab\": [round(float(value), 4) for value in lab],\n            \"hex\": rgb_to_hex(lab_to_rgb(lab)),\n            \"share\": round(float(counts[index] / len(model.labels_) * 100), 4),\n        })\n    rows.sort(key=lambda row: (-row[\"share\"], *row[\"lab\"]))\n    return rows\n\n\ndef compute_candidates(recipe_path, work_dir):\n    recipe = read_json(recipe_path)\n    source_path = obtain_source(recipe[\"source\"][\"image\"], work_dir)\n    _verify_source(recipe, source_path)\n    image = open_rgb(source_path)\n    candidates = []\n    for region in recipe[\"regions\"]:\n        rows = _cluster_region(image, region[\"box\"], region[\"k\"],\n                               recipe[\"extraction\"])\n        for number, row in enumerate(rows, start=1):\n            row.update({\n                \"id\": f'{region[\"id\"]}:c{number:02d}',\n                \"region\": region[\"id\"],\n                \"region_label\": region[\"label\"],\n            })\n            candidates.append(row)\n    return candidates\n\n\ndef save_candidates(recipe_path, work_dir, accept=True):\n    recipe_path = pathlib.Path(recipe_path)\n    work_dir = pathlib.Path(work_dir)\n    candidates = compute_candidates(recipe_path, work_dir)\n    write_json(work_dir / \"candidates.json\", {\"candidates\": candidates})\n    if accept:\n        recipe = read_json(recipe_path)\n        recipe[\"accepted_candidates\"] = candidates\n        write_json(recipe_path, recipe)\n    candidate_sheet(candidates, work_dir / \"candidates.png\")\n    return candidates\n\n\ndef candidate_sheet(candidates, output=None):\n    use_arial()\n    region_names = []\n    for candidate in candidates:\n        if candidate[\"region_label\"] not in region_names:\n            region_names.append(candidate[\"region_label\"])\n    rows = []\n    for name in region_names:\n        rows.append([item for item in candidates if item[\"region_label\"] == name])\n    columns = max(len(row) for row in rows)\n    fig, axes = plt.subplots(len(rows), 1, figsize=(max(10, columns * 1.35),\n                                                   max(2.1, len(rows) * 1.9)),\n                             squeeze=False)\n    for ax, name, row in zip(axes[:, 0], region_names, rows):\n        ax.set_xlim(0, columns)\n        ax.set_ylim(0, 1)\n        ax.axis(\"off\")\n        ax.text(-0.02, 0.5, name, ha=\"right\", va=\"center\", fontsize=10)\n        for column, candidate in enumerate(row):\n            ax.add_patch(Rectangle((column, 0.28), 0.95, 0.62,\n                                   facecolor=candidate[\"hex\"], edgecolor=\"white\"))\n            ax.text(column + 0.475, 0.18, candidate[\"id\"].split(\":\")[-1],\n                    ha=\"center\", va=\"center\", fontsize=8)\n            ax.text(column + 0.475, 0.05, candidate[\"hex\"],\n                    ha=\"center\", va=\"center\", fontsize=8)\n    fig.suptitle(\"K-means candidates, largest cluster first\", fontsize=14)\n    fig.tight_layout()\n    if output:\n        output = pathlib.Path(output)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(output, facecolor=\"white\", bbox_inches=\"tight\")\n    return fig\n\n\ndef save_curation(recipe_path, selections):\n    recipe = read_json(recipe_path)\n    candidates = {item[\"id\"]: item for item in recipe.get(\"accepted_candidates\", [])}\n    colors = []\n    for number, selection in enumerate(selections, start=1):\n        candidate_id = selection[\"candidate\"]\n        if candidate_id not in candidates:\n            raise ValueError(f\"candidate was not found: {candidate_id}\")\n        candidate = candidates[candidate_id]\n        colors.append({\n            \"id\": f\"p{number:02d}\",\n            \"from\": candidate_id,\n            \"source_hex\": candidate[\"hex\"],\n            \"current\": candidate[\"hex\"],\n            \"note\": str(selection.get(\"note\", \"\")).strip(),\n        })\n    if not 5 <= len(colors) <= 12:\n        raise ValueError(\"choose from 5 to 12 colors\")\n    recipe[\"curation\"] = {\n        \"colors\": colors,\n        \"operations\": [],\n        \"final_order\": [item[\"id\"] for item in colors],\n    }\n    recipe[\"expected\"] = {\"colors\": [item[\"current\"] for item in colors]}\n    write_json(recipe_path, recipe)\n    return recipe\n\n\ndef _apply_one(color, adjustment):\n    if \"replace\" in adjustment:\n        replacement = str(adjustment[\"replace\"]).lower()\n        if not re.fullmatch(r\"#[0-9a-f]{6}\", replacement):\n            raise ValueError(f\"bad replacement color: {replacement}\")\n        return replacement, \"replace\", {\"hex\": replacement}\n    delta = adjustment.get(\"delta\", {})\n    changes = {key: float(delta.get(key, 0)) for key in (\"L\", \"C\", \"H\")}\n    return apply_edit(color, changes), \"adjust_lch\", changes\n\n\ndef apply_adjustments(recipe_path, adjustments, output=None, reset=False):\n    recipe = read_json(recipe_path)\n    curation = recipe.get(\"curation\")\n    if not curation:\n        raise ValueError(\"save the candidate choices before adjusting colors\")\n    if reset:\n        for item in curation[\"colors\"]:\n            item[\"current\"] = item[\"source_hex\"]\n        curation[\"operations\"] = []\n    before_palette = [item[\"current\"] for item in curation[\"colors\"]]\n    by_id = {item[\"id\"]: item for item in curation[\"colors\"]}\n    for adjustment in adjustments:\n        target = adjustment[\"target\"]\n        if target not in by_id:\n            raise ValueError(f\"palette color was not found: {target}\")\n        reason = str(adjustment.get(\"reason\", \"\")).strip()\n        if not reason:\n            raise ValueError(f\"{target}: give a short reason for the adjustment\")\n        item = by_id[target]\n        before = item[\"current\"]\n        after, operation, detail = _apply_one(before, adjustment)\n        record = {\n            \"id\": f'a{len(curation[\"operations\"]) + 1:03d}',\n            \"op\": operation,\n            \"target\": target,\n            \"before\": before,\n            \"after\": after,\n            \"reason\": reason,\n        }\n        record.update(detail)\n        curation[\"operations\"].append(record)\n        item[\"current\"] = after\n    after_palette = [item[\"current\"] for item in curation[\"colors\"]]\n    recipe[\"expected\"] = {\"colors\": after_palette}\n    write_json(recipe_path, recipe)\n    before_after_sheet(before_palette, after_palette, output)\n    return recipe\n\n\ndef before_after_sheet(before, after, output=None):\n    use_arial()\n    fig, axes = plt.subplots(2, 1, figsize=(max(9, len(before) * 1.25), 2.8))\n    for ax, colors, label in zip(axes, (before, after), (\"Before\", \"After\")):\n        ax.set_xlim(0, len(colors))\n        ax.set_ylim(0, 1)\n        ax.axis(\"off\")\n        ax.text(-0.1, 0.5, label, ha=\"right\", va=\"center\")\n        for index, color in enumerate(colors):\n            ax.add_patch(Rectangle((index, 0), 1, 1,\n                                   facecolor=color, edgecolor=\"white\"))\n            ax.text(index + 0.5, -0.12, color, ha=\"center\", va=\"top\", fontsize=8)\n    fig.tight_layout()\n    if output:\n        output = pathlib.Path(output)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(output, facecolor=\"white\", bbox_inches=\"tight\")\n    return fig\n\n\ndef check_recipe(recipe_path, work_dir, output=None):\n    recipe = read_json(recipe_path)\n    colors = recipe.get(\"expected\", {}).get(\"colors\", [])\n    if len(colors) < 2:\n        raise ValueError(\"curate at least two colors before checking the recipe\")\n    source_path = obtain_source(recipe[\"source\"][\"image\"], work_dir)\n    _verify_source(recipe, source_path)\n    viewing = {}\n    for kind in VISION_TYPES:\n        minimum, mean = pairwise_min_mean(colors, kind)\n        viewing[VISION_LABELS[kind]] = {\n            \"minimum\": round(minimum, 4),\n            \"mean\": round(mean, 4),\n        }\n    presence = {\n        color: {\"nearest\": round(values[0], 4), \"share_within_8\": round(values[1], 4)}\n        for color, values in presence_in_image(colors, source_path).items()\n    }\n    report = {\n        \"palette\": recipe[\"palette\"],\n        \"colors\": colors,\n        \"suggested_pick_order\": greedy_order(colors),\n        \"colorblind\": worst_case(colors) >= COLORBLIND_THRESHOLD,\n        \"viewing\": viewing,\n        \"source_presence\": presence,\n    }\n    if output:\n        write_json(output, report)\n    return report\n\n\ndef palette_draft(recipe_path, metadata, output):\n    recipe = read_json(recipe_path)\n    curation = recipe.get(\"curation\")\n    if not curation:\n        raise ValueError(\"curate the colors before writing a palette draft\")\n    colors = recipe[\"expected\"][\"colors\"]\n    draft = {}\n    for key in (\"name\", \"persian\", \"pronunciation\", \"position\", \"about\",\n                \"story\", \"craft\", \"samples\"):\n        if key in metadata:\n            draft[key] = metadata[key]\n    draft[\"name\"] = metadata.get(\"name\", recipe[\"palette\"])\n    draft[\"colors\"] = colors\n    draft[\"notes\"] = [item[\"note\"] for item in curation[\"colors\"]]\n    draft[\"order\"] = greedy_order(colors)\n    draft[\"colorblind\"] = worst_case(colors) >= COLORBLIND_THRESHOLD\n    if \"source\" not in metadata:\n        raise ValueError(\"palette metadata must include the source record\")\n    draft[\"source\"] = metadata[\"source\"]\n    write_json(output, draft)\n    return draft\n\n\ndef verify_recipe(recipe_path, work_dir):\n    recipe = read_json(recipe_path)\n    accepted = recipe.get(\"accepted_candidates\", [])\n    fresh = compute_candidates(recipe_path, work_dir)\n    candidate_match = [item[\"id\"] for item in accepted] == [item[\"id\"] for item in fresh]\n    hex_match = [item[\"hex\"] for item in accepted] == [item[\"hex\"] for item in fresh]\n    centers_match = (len(accepted) == len(fresh)\n                     and all(np.allclose(saved[\"lab\"], current[\"lab\"], atol=0.02)\n                             for saved, current in zip(accepted, fresh)))\n    shares_match = (len(accepted) == len(fresh)\n                    and all(abs(saved[\"share\"] - current[\"share\"]) <= 0.02\n                            for saved, current in zip(accepted, fresh)))\n    curation = recipe.get(\"curation\", {})\n    state = {item[\"id\"]: item[\"source_hex\"] for item in curation.get(\"colors\", [])}\n    replay_errors = []\n    for operation in curation.get(\"operations\", []):\n        target = operation[\"target\"]\n        if state.get(target) != operation[\"before\"]:\n            replay_errors.append(f'{operation[\"id\"]}: before color does not match')\n            continue\n        if operation[\"op\"] == \"replace\":\n            after = operation[\"hex\"]\n        else:\n            after = apply_edit(state[target], operation)\n        if after != operation[\"after\"]:\n            replay_errors.append(f'{operation[\"id\"]}: after color does not replay')\n        state[target] = after\n    order = curation.get(\"final_order\", [])\n    replayed = [state[item] for item in order if item in state]\n    expected = recipe.get(\"expected\", {}).get(\"colors\", [])\n    return {\n        \"candidate_ids_match\": candidate_match,\n        \"candidate_hex_match\": hex_match,\n        \"candidate_centers_match\": centers_match,\n        \"candidate_shares_match\": shares_match,\n        \"replayed_colors_match\": replayed == expected,\n        \"replay_errors\": replay_errors,\n        \"verified\": candidate_match and hex_match and centers_match\n                    and shares_match and replayed == expected and not replay_errors,\n    }\n"}
for module_name in ("colorlib", "adjust_colors", "notebook_workflow"):
    module = types.ModuleType(module_name)
    module.__file__ = f"{module_name}.py"
    sys.modules[module_name] = module
    exec(compile(MODULE_SOURCES[module_name], module.__file__, "exec"),
         module.__dict__)

from notebook_workflow import *

PALETTE_SLUG = palette_slug(PALETTE_SLUG)
if IN_COLAB:
    WORK_DIR = Path("/content/rang-workflow") / PALETTE_SLUG
else:
    WORK_DIR = Path.cwd() / "cache" / "notebook-upload" / PALETTE_SLUG
WORK_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_PATH = WORK_DIR / f"{PALETTE_SLUG}-recipe.json"
use_arial()

def receive_file(local_path, label):
    if IN_COLAB:
        print(f"Choose {label} from your computer")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one file")
        filename, data = next(iter(uploaded.items()))
        output = WORK_DIR / Path(filename).name
        output.write_bytes(data)
        return output
    if not local_path:
        raise ValueError(f"Enter a local path for {label}")
    path = Path(local_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def load_workflow_zip(local_path):
    archive = receive_file(local_path, "the workflow ZIP from the previous notebook")
    with zipfile.ZipFile(archive) as handle:
        total = 0
        for item in handle.infolist():
            member = Path(item.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError("The workflow ZIP contains an unsafe path")
            total += item.file_size
        if total > 200 * 1024 * 1024:
            raise ValueError("The workflow ZIP expands beyond 200 MB")
        handle.extractall(WORK_DIR)
    if not RECIPE_PATH.exists():
        raise FileNotFoundError("The uploaded ZIP does not contain the expected recipe")
    return archive

def make_workflow_zip():
    archive = WORK_DIR / f"{PALETTE_SLUG}-workflow.zip"
    members = [path for path in WORK_DIR.iterdir()
               if path.is_file() and path.suffix.lower() != ".zip"]
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
        for member in sorted(members):
            handle.write(member, member.name)
    return archive

def offer_download(path):
    print("Saved:", path)
    if IN_COLAB:
        files.download(str(path))

print("Working folder:", WORK_DIR)

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Upload the workflow ZIP downloaded from the previous notebook. Local Jupyter users can enter its path.</div>

In [ ]:
LOCAL_WORKFLOW_ZIP = "" #@param {type:"string"}
load_workflow_zip(LOCAL_WORKFLOW_ZIP)
recipe = read_json(RECIPE_PATH)
print(f'Loaded {recipe["palette"]} with {len(recipe["regions"])} regions')

In [ ]:
result = verify_recipe(RECIPE_PATH, WORK_DIR)
for key, value in result.items():
    print(f"{key}: {value}")
if not result["verified"]:
    raise ValueError("The saved workflow did not reproduce exactly")
print("The recipe reproduced the accepted candidates and final colors.")

Verification is mechanical. It confirms that the saved image, regions,
k-means settings, choices, and adjustments still lead to the accepted colors.
It does not claim that another artist would make the same choices.

<div style="font-family:Arial,sans-serif;background:#d9edf7;padding:14px"><strong>SAVE YOUR WORK</strong><br>Download the final workflow ZIP for your contribution files.</div>

In [ ]:
workflow_zip = make_workflow_zip()
if DOWNLOAD_UPDATED_ZIP:
    offer_download(workflow_zip)
else:
    print("Workflow ZIP:", workflow_zip)